In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/titanic/train.csv
/kaggle/input/titanic/test.csv
/kaggle/input/titanic/gender_submission.csv


In [2]:

train_data = pd.read_csv("/kaggle/input/titanic/train.csv")
# train_data = train_data.dropna(subset=['Survived','Pclass','Age','SibSp','Parch','Embarked'])
# print(train_data.head())

test_data = pd.read_csv("/kaggle/input/titanic/test.csv")
# test_data = test_data.dropna(subset=['Pclass','Age','SibSp','Parch','Embarked'])
# print(test_data.head())



In [3]:
# print(train_data.dtypes)
# print(np.isinf(full_train.select_dtypes("number")).sum())

In [4]:
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

# Random forest
train = train_data.drop(['PassengerId','Name','Ticket','Cabin'], axis=1)
X_test = test_data.drop(['PassengerId','Name','Ticket','Cabin'], axis=1)

train['Sex'] = pd.factorize(train['Sex'])[0] + 1
train['Embarked'] = pd.factorize(train['Embarked'])[0] + 1
# print(train.head())
X_test['Sex'] = pd.factorize(X_test['Sex'])[0] + 1
X_test['Embarked'] = pd.factorize(X_test['Embarked'])[0] + 1
# print(X_test.head())

print('Original NaN values:')
print(train.isna().sum())
print(X_test.isna().sum())

imp = IterativeImputer(max_iter=10, random_state=0)
imp.fit(train)
imputed_train = pd.DataFrame(data = (imp.transform(train)),columns = ['Survived','Pclass','Age','Sex','SibSp','Parch','Fare','Embarked'])

imp = IterativeImputer(max_iter=10, random_state=0)
imp.fit(X_test)
imputed_X_test = pd.DataFrame(data = (imp.transform(X_test)), columns = ['Pclass','Age','Sex','SibSp','Parch','Fare','Embarked'])

print('\nAfter imputing:')
print(imputed_train.isna().sum())
print(imputed_X_test.isna().sum())

imputed_train

X = imputed_train.drop(['Survived'], axis=1)
y = imputed_train['Survived']


Original NaN values:
Survived      0
Pclass        0
Sex           0
Age         177
SibSp         0
Parch         0
Fare          0
Embarked      0
dtype: int64
Pclass       0
Sex          0
Age         86
SibSp        0
Parch        0
Fare         1
Embarked     0
dtype: int64

After imputing:
Survived    0
Pclass      0
Age         0
Sex         0
SibSp       0
Parch       0
Fare        0
Embarked    0
dtype: int64
Pclass      0
Age         0
Sex         0
SibSp       0
Parch       0
Fare        0
Embarked    0
dtype: int64


In [5]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import make_classification

clf = RandomForestClassifier(n_estimators = 1000, random_state=0)


In [6]:
clf.fit(X, y)

RandomForestClassifier(n_estimators=1000, random_state=0)

In [7]:
rf_pred = clf.predict(imputed_X_test).astype(int)

Second model XGBoost

In [8]:
# XGBoost
from sklearn.datasets import make_hastie_10_2
from sklearn.ensemble import GradientBoostingClassifier

clf_GB = GradientBoostingClassifier(n_estimators=1000, learning_rate=0.1,
    max_depth=1, random_state=0)
clf_GB.fit(X,y)

GB_pred = clf_GB.predict(imputed_X_test).astype(int)
GB_pred

array([0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 1, 0, 1, 1, 0, 0, 1, 0, 1, 0,
       1, 1, 1, 0, 1, 0, 0, 0, 0, 0, 1, 1, 1, 0, 1, 1, 0, 0, 0, 0, 0, 1,
       1, 0, 0, 0, 1, 1, 1, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1, 1,
       1, 0, 1, 1, 1, 0, 1, 0, 1, 0, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 1, 1,
       1, 1, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0,
       0, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1,
       0, 0, 1, 1, 0, 1, 1, 0, 1, 0, 0, 1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1,
       1, 0, 1, 1, 0, 1, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 1,
       0, 1, 1, 1, 0, 1, 0, 1, 1, 0, 1, 0, 0, 0, 1, 1, 0, 0, 1, 0, 1, 0,
       1, 0, 1, 0, 1, 1, 0, 1, 0, 0, 1, 1, 0, 0, 1, 0, 0, 0, 1, 1, 1, 1,
       0, 0, 0, 0, 1, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1,
       0, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 1, 0, 0, 0, 0, 1, 1, 1, 1, 1, 0,
       0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0,

In [9]:
rf_pred = pd.DataFrame({"Survived": rf_pred})
predictions_rf = pd.concat([test_data['PassengerId'], rf_pred], axis=1)
predictions_rf.to_csv('submission.csv', index=False)





**Accuracy of predicting survivors in hidden test is 0.77990.
This test data is not usable in training.**
